# ReaVector — build model weights in PARTS (fits Kaggle disk)

Each run downloads **one part** (<= ~10 GB), writes a `part.json`, and publishes it as a
private Dataset. The worker mounts all parts and merges them (symlinks).

### One-time setup (auto-publish)
Kaggle → **Add-ons → Secrets** → add `KAGGLE_USERNAME` and `KAGGLE_KEY` (from your kaggle.json).
Then this notebook creates the dataset automatically. Without secrets, it tells you to publish manually.

### For each part
Set `PART_INDEX = 0`, Run All, then `1`, `2`, ... (editing only that one number).
Send me the dataset slugs afterwards; I mount them in the worker.

Default plan below is **Z-Image-Turbo (4 parts)**. To do another model, change `MODEL_ID`,
`REPO`, and `PARTS` (see the tables at the bottom of this cell).

In [ ]:
!pip -q install -U huggingface_hub kaggle

In [ ]:
import os, json
from huggingface_hub import snapshot_download

os.environ['HF_HUB_DISABLE_XET'] = '1'   # avoid temp duplication -> fits the disk

MODEL_ID = 'z-image-turbo'
REPO = 'Tongyi-MAI/Z-Image-Turbo'
SLUG_PREFIX = 'reavector-zimage'

PARTS = [
    ['transformer/diffusion_pytorch_model-00001-of-00003.safetensors'],
    ['transformer/diffusion_pytorch_model-00002-of-00003.safetensors'],
    ['transformer/diffusion_pytorch_model-00003-of-00003.safetensors', 'transformer/config.json'],
    ['text_encoder/*', 'vae/*', 'tokenizer/*', 'scheduler/*', 'model_index.json'],
]
PART_INDEX = 0

DEST = '/kaggle/working/part'
os.makedirs(DEST, exist_ok=True)
print('downloading part', PART_INDEX, 'of', MODEL_ID, PARTS[PART_INDEX], flush=True)
snapshot_download(repo_id=REPO, local_dir=DEST, allow_patterns=PARTS[PART_INDEX])
json.dump({'model_id': MODEL_ID, 'repo': REPO, 'part': PART_INDEX, 'patterns': PARTS[PART_INDEX]},
          open(os.path.join(DEST, 'part.json'), 'w'), indent=2)
!du -sh /kaggle/working/part
!find /kaggle/working/part -type f | head -30

In [ ]:
import os, json
try:
    from kaggle_secrets import UserSecretsClient
    s = UserSecretsClient()
    os.environ['KAGGLE_USERNAME'] = s.get_secret('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = s.get_secret('KAGGLE_KEY')
    print('secrets loaded')
except Exception as e:
    print('secrets NOT configured:', e)
user = os.environ.get('KAGGLE_USERNAME', '')
slug = f'{SLUG_PREFIX}-p{PART_INDEX+1}'
meta = {'title': slug, 'id': f'{user}/{slug}', 'licenses': [{'name': 'other'}]}
json.dump(meta, open(os.path.join(DEST, 'dataset-metadata.json'), 'w'), indent=2)
print('metadata:', meta)
if user:
    !cd /kaggle/working && kaggle datasets create -p /kaggle/working/part || echo 'create failed (may already exist)'
else:
    print('No secrets. Publish manually: Output -> part -> New Dataset, name it', slug)

## Other models (change MODEL_ID/REPO/PARTS)

**Wan 2.1 1.3B** (`wan2.1-1.3b`, repo `Wan-AI/Wan2.1-T2V-1.3B-Diffusers`):
- p1 `text_encoder/model-00001-of-00005.safetensors`
- p2 `text_encoder/model-00002-of-00005.safetensors`
- p3 `text_encoder/model-00003-of-00005.safetensors`
- p4 `text_encoder/model-00004-of-00005.safetensors`
- p5 `text_encoder/model-00005-of-00005.safetensors` + its index/config
- p6 `transformer/*`, `vae/*`, `scheduler/*`, `tokenizer/*`, `model_index.json`

**SDXL** (`sdxl-base-1.0`, also used by `texture`): repo `stabilityai/stable-diffusion-xl-base-1.0`
- p1 `text_encoder/*`
- p2 `text_encoder_2/*`
- p3 `unet/*`
- p4 `vae/*`, `tokenizer/*`, `tokenizer_2/*`, `scheduler/*`, `model_index.json`

Keep each part <= ~10 GB. With `HF_HUB_DISABLE_XET=1` a single ~10 GB file fits.